# Notebook 9 — Hybrid Model 3: Entropy-Guided Stochastic Rebalancing

Combines:
1. Class-cap enforcement
2. Entropy maximisation
3. Seeded stochastic selection
4. Global distribution lock

Sensitive attribute: `fitness_level`

In [11]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json, os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('../data', exist_ok=True)
os.makedirs('../results', exist_ok=True)

df          = pd.read_csv('../data/t_close_anonymized.csv')
df_original = pd.read_csv('../data/raw_fitness_data.csv')

quasi_identifiers   = ['age', 'region', 'activity']
sensitive_attribute = 'fitness_level'
fitness_classes     = ['fit', 'moderately_fit', 'unfit']
t_threshold         = 0.25

CLASS_CAP      = 0.55
ENTROPY_TARGET = 0.95
SEED           = 42

assert 'fitness_level' in df.columns, 'Run previous notebooks first.'

print("Data loaded:", df.shape)
print(df['fitness_level'].value_counts())

Data loaded: (7000, 11)
fitness_level
moderately_fit    4015
fit               1504
unfit             1481
Name: count, dtype: int64


In [12]:
pop_fl_dist = df_original['fitness_level'].value_counts(normalize=True).reindex(fitness_classes, fill_value=0)

def tvd(p, q):
    p = p.reindex(q.index, fill_value=0)
    return 0.5 * sum(abs(p[i] - q[i]) for i in q.index)

def normalised_entropy(dist):
    probs = np.array([dist.get(c, 0) for c in fitness_classes])
    probs = probs[probs > 0]
    if len(probs) == 0:
        return 0.0
    H = -np.sum(probs * np.log(probs))
    return H / np.log(len(fitness_classes))

def inference_attack_accuracy(df_in):
    correct = 0
    for _, g in df_in.groupby(quasi_identifiers):
        pred = g['fitness_level'].mode().iloc[0]
        correct += (g['fitness_level'] == pred).sum()
    return correct / len(df_in)

acc_before = inference_attack_accuracy(df)
print("Baseline accuracy:", acc_before)

Baseline accuracy: 0.6792857142857143


In [13]:
rng = np.random.default_rng(SEED)

df_h3 = df.copy()

pass1_adj = 0
pass2_adj = 0
pass3_adj = 0

for _, g in df_h3.groupby(quasi_identifiers):

    if len(g) < 3:
        continue

    # PASS 1 — Class cap enforcement
    for _ in range(5):
        dist = g['fitness_level'].value_counts(normalize=True).reindex(fitness_classes, fill_value=0)
        over = dist[dist > CLASS_CAP]

        if over.empty:
            break

        over_class = over.idxmax()
        under_class = (pop_fl_dist - dist).idxmax()

        idx = g[g['fitness_level'] == over_class].index
        n_fix = int((dist[over_class] - CLASS_CAP) * len(g))

        chosen = idx[:max(1, n_fix)]
        df_h3.loc[chosen, 'fitness_level'] = under_class
        pass1_adj += len(chosen)

    # PASS 2 — Entropy boost
    dist = df_h3.loc[g.index]['fitness_level'].value_counts(normalize=True)
    ent = normalised_entropy(dist)

    if ent < ENTROPY_TARGET:
        under = (pop_fl_dist - dist).idxmax()
        over = (pop_fl_dist - dist).idxmin()

        idx = df_h3.loc[g.index][df_h3.loc[g.index]['fitness_level'] == over].index
        if len(idx) > 0:
            pick = rng.choice(idx)
            df_h3.loc[pick, 'fitness_level'] = under
            pass2_adj += 1

    # PASS 3 — TVD correction
    dist = df_h3.loc[g.index]['fitness_level'].value_counts(normalize=True).reindex(fitness_classes, fill_value=0)
    if tvd(dist, pop_fl_dist) > t_threshold:
        under = (pop_fl_dist - dist).idxmax()
        over = (pop_fl_dist - dist).idxmin()

        idx = df_h3.loc[g.index][df_h3.loc[g.index]['fitness_level'] == over].index
        if len(idx) > 0:
            df_h3.loc[idx[0], 'fitness_level'] = under
            pass3_adj += 1

print("Adjustments:", pass1_adj, pass2_adj, pass3_adj)

Adjustments: 4495 131 33


In [14]:
viol = 0
tvd_vals = []

for _, g in df_h3.groupby(quasi_identifiers):
    d = tvd(g['fitness_level'].value_counts(normalize=True), pop_fl_dist)
    tvd_vals.append(d)
    if d > t_threshold:
        viol += 1

acc_after = inference_attack_accuracy(df_h3)

print("Before:", acc_before)
print("After:", acc_after)
print("Reduction:", acc_before - acc_after)
print("Violations:", viol)
print("Mean TVD:", np.mean(tvd_vals))

Before: 0.6792857142857143
After: 0.5441428571428572
Reduction: 0.13514285714285712
Violations: 31
Mean TVD: 0.1616774102510508


In [15]:
# Save dataset (same style as others)
df_h3.to_csv('../data/hybrid3_result.csv', index=False)
print("✅ Saved: ../data/hybrid3_result.csv")

✅ Saved: ../data/hybrid3_result.csv


In [16]:
h3_metrics = {
    "method": "Hybrid 3 — Entropy-Guided Stochastic Rebalancing",
    "sensitive_attribute": "fitness_level",
    "inference_attack_accuracy": float(acc_after),
    "inference_attack_accuracy_before": float(acc_before),
    "attack_accuracy_reduction": float(acc_before - acc_after),
    "violations_after": int(viol),
    "mean_tvd": float(np.mean(tvd_vals)),
    "total_adjustments": int(pass1_adj + pass2_adj + pass3_adj),
    "pass1_class_cap": int(pass1_adj),
    "pass2_entropy": int(pass2_adj),
    "pass3_tvd": int(pass3_adj),
    "data_loss_percent": 0.0
}

with open('../results/hybrid3_metrics.json', 'w') as f:
    json.dump(h3_metrics, f, indent=2)

print("\nHybrid 3 summary:")
for k, v in h3_metrics.items():
    print(f"  {k}: {v}")


Hybrid 3 summary:
  method: Hybrid 3 — Entropy-Guided Stochastic Rebalancing
  sensitive_attribute: fitness_level
  inference_attack_accuracy: 0.5441428571428572
  inference_attack_accuracy_before: 0.6792857142857143
  attack_accuracy_reduction: 0.13514285714285712
  violations_after: 31
  mean_tvd: 0.1616774102510508
  total_adjustments: 4659
  pass1_class_cap: 4495
  pass2_entropy: 131
  pass3_tvd: 33
  data_loss_percent: 0.0


In [19]:
# -------- VISUALIZATION (same style as Hybrid 2) --------

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# TVD distribution
tvd_before = [
    tvd(g['fitness_level'].value_counts(normalize=True), pop_fl_dist)
    for _, g in df.groupby(quasi_identifiers)
]

axes[0].hist(tvd_before, bins=20, alpha=0.6, label='Before', color='red')
axes[0].hist(tvd_vals, bins=20, alpha=0.6, label='After', color='green')
axes[0].axvline(x=t_threshold, linestyle='--', color='blue', label=f't={t_threshold}')
axes[0].set_title('TVD Distribution')
axes[0].legend()

# Class distribution
before_counts = df['fitness_level'].value_counts()
after_counts  = df_h3['fitness_level'].value_counts()

x = np.arange(len(fitness_classes))
axes[1].bar(x - 0.2, before_counts.reindex(fitness_classes), 0.4, label='Before')
axes[1].bar(x + 0.2, after_counts.reindex(fitness_classes), 0.4, label='After')
axes[1].set_xticks(x)
axes[1].set_xticklabels(fitness_classes)
axes[1].set_title('Fitness Level Distribution')
axes[1].legend()

# Attack accuracy
axes[2].bar(['Before', 'After'], [acc_before, acc_after])
axes[2].set_title('Inference Attack Accuracy')

for i, v in enumerate([acc_before, acc_after]):
    axes[2].text(i, v + 0.01, f"{v:.3f}", ha='center')

plt.tight_layout()
plt.savefig('../results/hybrid3_analysis.png')
plt.show()
plt.close()

print("✅ Saved: ../results/hybrid3_analysis.png")

✅ Saved: ../results/hybrid3_analysis.png


In [10]:
df_h3.to_csv('../data/hybrid3_result.csv', index=False)

metrics = {
    "accuracy_before": float(acc_before),
    "accuracy_after": float(acc_after),
    "reduction": float(acc_before - acc_after),
    "violations": int(viol),
    "mean_tvd": float(np.mean(tvd_vals))
}

with open('../results/hybrid3_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Saved results.")

Saved results.
